# Chapter 1: The Singular Value Decomposition

## Data-Driven Science and Engineering — Review Notebook

This notebook combines concise notes with self-contained Python versions of the computational examples in Chapter 1. It follows the official companion-code sequence through matrix approximation, least-squares regression, PCA, eigenimages, denoising, randomized SVD, and tensors.

**How to use it:** read the short notes, predict what each code cell will show, run it, and then summarize the numerical or visual result in your own words.

| Section | Main question | Computational example |
|---|---|---|
| 1.1 | What does the SVD do geometrically? | Map a circle into an ellipse |
| 1.2 | How can a matrix be approximated efficiently? | Low-rank image compression |
| 1.3 | Which mathematical properties make the SVD useful? | Correlation matrices and unitary invariance |
| 1.4 | How does the SVD solve regression problems? | Line, cement, and multivariable regression |
| 1.5 | How is PCA obtained from the SVD? | Gaussian and medical-data PCA |
| 1.6 | How do data-adapted image bases work? | Eigenimage reconstruction and coordinates |
| 1.7 | How does low rank support denoising? | Singular-value thresholding and rotated images |
| 1.8 | How can a large SVD be approximated? | Randomized SVD and power iterations |
| 1.9 | How does the idea extend beyond matrices? | A low-rank space-time tensor |

The notes are original summaries. The examples are modernized from the official companion notebooks at [databookuw.com](https://www.databookuw.com/) and are designed to run without the book's separate data folder.

# 1.1 Overview: the big picture

For any matrix $X\in\mathbb{R}^{m\times n}$,

$$
X=U\Sigma V^T.
$$

- $U\in\mathbb{R}^{m\times r}$ contains orthonormal **left singular vectors**.
- $V\in\mathbb{R}^{n\times r}$ contains orthonormal **right singular vectors**.
- $\Sigma=\operatorname{diag}(\sigma_1,\ldots,\sigma_r)$ contains nonnegative singular values in descending order.
- $r=\operatorname{rank}(X)$ in the compact mathematical decomposition. Numerical routines often return $\min(m,n)$ triplets, including values that are effectively zero.

The SVD separates a linear map into three actions:

$$
x\xrightarrow{V^T}\text{rotate/reflection}
\xrightarrow{\Sigma}\text{axis scaling}
\xrightarrow{U}\text{rotate/reflection}.
$$

A unit circle is therefore sent to an ellipse. The ellipse's semiaxis lengths are the singular values, its input directions are the columns of $V$, and its output directions are the columns of $U$.

The same factorization supports compression, denoising, least squares, PCA, coordinate discovery, and fast approximate algorithms. Its central idea is to rank directions by how strongly the data or linear transformation acts along them.

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.patches import Ellipse
from scipy.ndimage import rotate
from sklearn.datasets import load_breast_cancer, load_diabetes, load_digits
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)
np.set_printoptions(precision=4, suppress=True)

SEED = 42
rng = np.random.default_rng(SEED)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

In [ ]:
# Geometric interpretation: a matrix maps the unit circle to an ellipse.
A_geometry = np.array([[3.0, 1.0], [1.0, 2.0]])
U_geometry, s_geometry, Vt_geometry = np.linalg.svd(A_geometry)

theta = np.linspace(0, 2 * np.pi, 500)
circle = np.vstack((np.cos(theta), np.sin(theta)))
ellipse = A_geometry @ circle

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(circle[0], circle[1], color="tab:blue", lw=2)
axes[0].set(title="Input: unit circle", xlabel="$x_1$", ylabel="$x_2$", aspect="equal")

axes[1].plot(ellipse[0], ellipse[1], color="tab:orange", lw=2)
for length, direction, color in zip(s_geometry, U_geometry.T, ["tab:red", "tab:green"]):
    axes[1].quiver(0, 0, *(length * direction), angles="xy", scale_units="xy",
                   scale=1, color=color, width=0.015)
axes[1].set(title="Output: ellipse $A x$", xlabel="$y_1$", ylabel="$y_2$", aspect="equal")
plt.tight_layout()
plt.show()

print("Singular values:", s_geometry)
print("Reconstruction error ||A - UΣVᵀ||₂ =",
      np.linalg.norm(A_geometry - U_geometry @ np.diag(s_geometry) @ Vt_geometry, 2))
print("UᵀU =\n", U_geometry.T @ U_geometry)
print("VᵀV =\n", Vt_geometry @ Vt_geometry.T)

# 1.2 Matrix approximation

Expanding the compact SVD gives a sum of rank-one matrices:

$$
X=\sum_{k=1}^{r}\sigma_k u_kv_k^T.
$$

Keeping only the first $q$ terms produces the truncated SVD

$$
X_q=U_q\Sigma_qV_q^T.
$$

The **Eckart–Young theorem** says that $X_q$ is the best rank-$q$ approximation to $X$ in both the spectral and Frobenius norms:

$$
\|X-X_q\|_2=\sigma_{q+1},
\qquad
\|X-X_q\|_F^2=\sum_{k=q+1}^{r}\sigma_k^2.
$$

A dense $m\times n$ matrix stores $mn$ numbers. Its rank-$q$ representation stores approximately $q(m+n+1)$ numbers. Compression is useful only when $q$ is much smaller than both matrix dimensions.

The fraction of Frobenius energy captured by the first $q$ modes is

$$
\frac{\sum_{k=1}^{q}\sigma_k^2}{\sum_{k=1}^{r}\sigma_k^2}.
$$

In [ ]:
# Self-contained counterpart to the book's image-compression example.
# A mosaic of bundled handwritten digits acts as one grayscale image.
digits_bundle = load_digits()
tile_rows, tile_cols = 10, 20
digit_tiles = digits_bundle.images[: tile_rows * tile_cols]
digit_mosaic = np.block([
    [digit_tiles[i * tile_cols + j] for j in range(tile_cols)]
    for i in range(tile_rows)
])

U_image, s_image, Vt_image = np.linalg.svd(digit_mosaic, full_matrices=False)
image_ranks = [2, 5, 10, 20, 40]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()
axes[0].imshow(digit_mosaic, cmap="gray")
axes[0].set_title(f"Original ({digit_mosaic.shape[0]}×{digit_mosaic.shape[1]})")

for ax, rank in zip(axes[1:], image_ranks):
    approximation = (U_image[:, :rank] * s_image[:rank]) @ Vt_image[:rank]
    ax.imshow(approximation, cmap="gray", vmin=0, vmax=16)
    ax.set_title(f"Rank {rank}")

for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
image_energy = np.cumsum(s_image**2) / np.sum(s_image**2)
image_rows = []
for rank in image_ranks:
    approximation = (U_image[:, :rank] * s_image[:rank]) @ Vt_image[:rank]
    image_rows.append({
        "rank": rank,
        "energy captured": image_energy[rank - 1],
        "relative Frobenius error": np.linalg.norm(digit_mosaic - approximation, "fro")
                                    / np.linalg.norm(digit_mosaic, "fro"),
        "stored values": rank * (digit_mosaic.shape[0] + digit_mosaic.shape[1] + 1),
        "compression ratio": digit_mosaic.size
                             / (rank * (digit_mosaic.shape[0] + digit_mosaic.shape[1] + 1)),
    })

image_summary = pd.DataFrame(image_rows)
display(image_summary.round({
    "energy captured": 4,
    "relative Frobenius error": 4,
    "compression ratio": 2,
}))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].semilogy(s_image, "o-", ms=3)
axes[0].set(title="Singular-value spectrum", xlabel="Index", ylabel="$\\sigma_k$")
axes[1].plot(image_energy, lw=2)
axes[1].axhline(0.90, color="tab:red", ls="--", label="90% energy")
axes[1].set(title="Cumulative energy", xlabel="Retained modes", ylabel="Fraction", ylim=(0, 1.02))
axes[1].legend()
plt.tight_layout()
plt.show()

# 1.3 Mathematical properties

The singular vectors diagonalize the two correlation matrices:

$$
XX^T=U\Sigma^2U^T,
\qquad
X^TX=V\Sigma^2V^T.
$$

Thus $u_k$ is an eigenvector of $XX^T$, $v_k$ is an eigenvector of $X^TX$, and both have eigenvalue $\sigma_k^2$. If one side of $X$ is much smaller, it can be cheaper to eigendecompose the smaller correlation matrix. This is often called the **method of snapshots**.

Important consequences:

- $\operatorname{rank}(X)$ is the number of nonzero singular values.
- $\|X\|_2=\sigma_1$ and $\|X\|_F^2=\sum_k\sigma_k^2$.
- The 2-norm condition number is $\kappa_2(X)=\sigma_1/\sigma_r$ for a full-rank matrix.
- Orthogonal changes of coordinates do not change singular values: if $Q$ and $P$ are orthogonal, $QXP$ has the same spectrum as $X$.
- The Moore–Penrose pseudoinverse is $X^+=V\Sigma^+U^T$, where nonzero singular values are inverted.

In [ ]:
# Rotation and scaling, following the geometry used in the companion example.
angle = np.deg2rad(35)
rotation = np.array([[np.cos(angle), -np.sin(angle)],
                     [np.sin(angle),  np.cos(angle)]])
scaling = np.diag([3.0, 0.8])
transformation = rotation @ scaling

transformed_circle = transformation @ circle
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(circle[0], circle[1], label="Unit circle", lw=2)
ax.plot(transformed_circle[0], transformed_circle[1], label="Rotated/scaled ellipse", lw=2)
ax.set(xlabel="$x_1$", ylabel="$x_2$", aspect="equal",
       title="Orthogonal maps rotate; diagonal maps scale")
ax.legend()
plt.show()

print("Singular values of the transformation:", np.linalg.svd(transformation, compute_uv=False))
print("Expected axis scales:", np.diag(scaling))

In [ ]:
# Verify the correlation-matrix identities and orthogonal invariance.
X_property = rng.normal(size=(8, 5))
U_property, s_property, Vt_property = np.linalg.svd(X_property, full_matrices=False)

left_error = np.linalg.norm(
    X_property @ X_property.T
    - U_property @ np.diag(s_property**2) @ U_property.T
)
right_error = np.linalg.norm(
    X_property.T @ X_property
    - Vt_property.T @ np.diag(s_property**2) @ Vt_property
)

Q_left, _ = np.linalg.qr(rng.normal(size=(8, 8)))
Q_right, _ = np.linalg.qr(rng.normal(size=(5, 5)))
transformed_singular_values = np.linalg.svd(Q_left @ X_property @ Q_right, compute_uv=False)

print(f"Left correlation identity error:  {left_error:.2e}")
print(f"Right correlation identity error: {right_error:.2e}")
print("Largest change after orthogonal coordinate transformations:",
      np.max(np.abs(s_property - transformed_singular_values)))
print("Rank:", np.linalg.matrix_rank(X_property))
print("Spectral norm:", s_property[0])
print("Condition number:", s_property[0] / s_property[-1])

# 1.4 Linear systems, least squares, and regression

A linear model has the form

$$
A\beta\approx b,
$$

where the columns of $A$ are measured features or chosen basis functions, $b$ contains outcomes, and $\beta$ contains unknown coefficients.

When the system is overdetermined, ordinary least squares solves

$$
\hat\beta=\arg\min_\beta\|A\beta-b\|_2^2=A^+b.
$$

If $A=U\Sigma V^T$, then

$$
\hat\beta=V\Sigma^+U^Tb.
$$

The SVD exposes poorly determined directions: a small $\sigma_k$ causes $1/\sigma_k$ to amplify noise. In practice, `np.linalg.lstsq` or `np.linalg.pinv` is safer than explicitly forming an inverse. Centering or standardizing features also improves interpretation when their units differ.

In [ ]:
# Simple linear regression through the origin.
x_line = 3 * rng.random(25)
y_line = 2.5 * x_line + 0.9 * rng.standard_normal(x_line.size)
A_line = x_line[:, None]

U_line, s_line, Vt_line = np.linalg.svd(A_line, full_matrices=False)
slope_svd = (Vt_line.T @ ((U_line.T @ y_line) / s_line)).item()
slope_lstsq = np.linalg.lstsq(A_line, y_line, rcond=None)[0].item()

x_plot = np.linspace(0, 3, 200)
plt.figure(figsize=(7, 4.5))
plt.scatter(x_line, y_line, color="tab:blue", label="Noisy observations")
plt.plot(x_plot, slope_svd * x_plot, color="tab:red", lw=2,
         label=f"SVD fit: $y={slope_svd:.2f}x$")
plt.xlabel("$x$")
plt.ylabel("$y$")
plt.title("Least-squares line from the pseudoinverse")
plt.legend()
plt.show()

print(f"SVD solution:   {slope_svd:.8f}")
print(f"lstsq solution: {slope_lstsq:.8f}")

## Hald cement regression

The Hald data relate heat released by cement to four ingredient measurements. This small dataset is embedded directly below. The textbook companion example fits the four coefficients without adding an intercept, then compares the observed heat values with $A\hat\beta$.

Coefficients quantify association conditional on the other included columns; they should not automatically be interpreted as causal effects.

In [ ]:
cement_A = np.array([
    [7, 26, 6, 60], [1, 29, 15, 52], [11, 56, 8, 20],
    [11, 31, 8, 47], [7, 52, 6, 33], [11, 55, 9, 22],
    [3, 71, 17, 6], [1, 31, 22, 44], [2, 54, 18, 22],
    [21, 47, 4, 26], [1, 40, 23, 34], [11, 66, 9, 12],
    [10, 68, 8, 12],
], dtype=float)
cement_heat = np.array([
    78.5, 74.3, 104.3, 87.6, 95.9, 109.2, 102.7,
    72.5, 93.1, 115.9, 83.8, 113.3, 109.4,
])

cement_beta = np.linalg.lstsq(cement_A, cement_heat, rcond=None)[0]
cement_prediction = cement_A @ cement_beta

display(pd.DataFrame({
    "ingredient": ["x1", "x2", "x3", "x4"],
    "coefficient": cement_beta,
}).round({"coefficient": 4}))

plt.figure(figsize=(9, 4.5))
plt.plot(cement_heat, "o-", color="black", lw=2, label="Observed heat")
plt.plot(cement_prediction, "s--", color="tab:red", label="Least-squares prediction")
plt.xlabel("Cement mixture")
plt.ylabel("Heat")
plt.title("Hald cement regression")
plt.legend()
plt.show()

print(f"RMSE: {np.sqrt(np.mean((cement_heat - cement_prediction)**2)):.3f}")

## Multivariable regression and feature significance

The companion code next fits housing value from multiple neighborhood attributes and repeats the fit after standardizing predictors. The self-contained version below uses the bundled diabetes regression dataset, which has the same mathematical structure.

If feature $j$ is standardized as

$$
z_j=\frac{x_j-\bar x_j}{s_j},
$$

then its coefficient measures the expected change in the prediction for a one-standard-deviation increase in that feature, holding the others fixed. This makes coefficient magnitudes more comparable, but correlation among features can still make rankings unstable.

In [ ]:
diabetes = load_diabetes()
X_diabetes = StandardScaler().fit_transform(diabetes.data)
y_diabetes = diabetes.target
A_diabetes = np.column_stack((X_diabetes, np.ones(X_diabetes.shape[0])))
beta_diabetes = np.linalg.lstsq(A_diabetes, y_diabetes, rcond=None)[0]
prediction_diabetes = A_diabetes @ beta_diabetes
order_diabetes = np.argsort(y_diabetes)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(y_diabetes[order_diabetes], color="black", lw=2, label="Observed")
axes[0].plot(prediction_diabetes[order_diabetes], color="tab:red", lw=1.5, label="Predicted")
axes[0].set(title="Multivariable least-squares fit", xlabel="Samples sorted by outcome", ylabel="Outcome")
axes[0].legend()

coefficient_order = np.argsort(np.abs(beta_diabetes[:-1]))
axes[1].barh(np.array(diabetes.feature_names)[coefficient_order],
             beta_diabetes[:-1][coefficient_order], color="tab:blue")
axes[1].axvline(0, color="black", lw=1)
axes[1].set(title="Coefficients for standardized features", xlabel="Coefficient")
plt.tight_layout()
plt.show()

ss_res = np.sum((y_diabetes - prediction_diabetes)**2)
ss_tot = np.sum((y_diabetes - y_diabetes.mean())**2)
print(f"In-sample R²: {1 - ss_res / ss_tot:.3f}")
print(f"RMSE: {np.sqrt(np.mean((y_diabetes - prediction_diabetes)**2)):.2f}")

# 1.5 Principal component analysis (PCA)

PCA finds orthogonal directions that capture the greatest variance. Given observations stored as rows, first center the columns:

$$
X_c=X-\mathbf{1}\mu^T.
$$

Then compute

$$
X_c=U\Sigma V^T.
$$

- The columns of $V$ are the **principal directions** or loadings in feature space.
- The coordinates $Z=X_cV=U\Sigma$ are the **principal-component scores**.
- The sample covariance matrix is $C=X_c^TX_c/(n-1)=V\Sigma^2V^T/(n-1)$.
- Explained-variance ratios are $\sigma_k^2/\sum_j\sigma_j^2$.

Centering is essential. Standardization is additionally useful when feature scales or units differ. PCA is unsupervised: it preserves variance, not necessarily the information most useful for predicting a label.

In [ ]:
# PCA of a rotated two-dimensional Gaussian cloud.
covariance_true = rotation @ np.diag([3.0, 0.35]) @ rotation.T
gaussian_points = rng.multivariate_normal([2.0, -1.0], covariance_true, size=800)
gaussian_centered = gaussian_points - gaussian_points.mean(axis=0)
U_gaussian, s_gaussian, Vt_gaussian = np.linalg.svd(gaussian_centered, full_matrices=False)
gaussian_scores = gaussian_centered @ Vt_gaussian.T

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(*gaussian_points.T, s=10, alpha=0.35)
center = gaussian_points.mean(axis=0)
for k, color in enumerate(["tab:red", "tab:green"]):
    scale = 2 * s_gaussian[k] / np.sqrt(gaussian_points.shape[0] - 1)
    direction = Vt_gaussian[k]
    axes[0].quiver(*center, *(scale * direction), angles="xy", scale_units="xy",
                   scale=1, color=color, width=0.012)
axes[0].set(title="Data and principal directions", xlabel="$x_1$", ylabel="$x_2$", aspect="equal")

axes[1].scatter(*gaussian_scores.T, s=10, alpha=0.35, color="tab:purple")
axes[1].axhline(0, color="black", lw=0.8)
axes[1].axvline(0, color="black", lw=0.8)
axes[1].set(title="Data in principal coordinates", xlabel="PC1 score", ylabel="PC2 score", aspect="equal")
plt.tight_layout()
plt.show()

explained_gaussian = s_gaussian**2 / np.sum(s_gaussian**2)
print("Explained-variance ratios:", explained_gaussian)

## PCA of medical measurements

The official example applies PCA to ovarian-cancer measurements and visualizes whether known groups separate in low-dimensional coordinates. Here, the bundled breast-cancer dataset provides a portable analogue with 30 standardized measurements and two diagnostic labels.

The labels are used only to color the plot after PCA. They do not influence the principal directions. Visible separation therefore indicates that high-variance unsupervised directions happen to align with group differences.

In [ ]:
cancer = load_breast_cancer()
X_cancer = StandardScaler().fit_transform(cancer.data)
X_cancer -= X_cancer.mean(axis=0)
U_cancer, s_cancer, Vt_cancer = np.linalg.svd(X_cancer, full_matrices=False)
scores_cancer = U_cancer * s_cancer
explained_cancer = s_cancer**2 / np.sum(s_cancer**2)

fig = plt.figure(figsize=(14, 5))
ax1 = fig.add_subplot(121)
ax1.plot(np.arange(1, len(s_cancer) + 1), explained_cancer, "o-", label="Individual")
ax1.plot(np.arange(1, len(s_cancer) + 1), np.cumsum(explained_cancer), "s-", label="Cumulative")
ax1.set(title="PCA variance spectrum", xlabel="Principal component", ylabel="Explained-variance fraction", ylim=(0, 1.02))
ax1.legend()

ax2 = fig.add_subplot(122, projection="3d")
for label, name, color in [(0, "malignant", "tab:red"), (1, "benign", "tab:blue")]:
    mask = cancer.target == label
    ax2.scatter(scores_cancer[mask, 0], scores_cancer[mask, 1], scores_cancer[mask, 2],
                s=18, alpha=0.65, label=name, color=color)
ax2.set(title="First three PCA scores", xlabel="PC1", ylabel="PC2", zlabel="PC3")
ax2.legend()
plt.tight_layout()
plt.show()

display(pd.DataFrame({
    "component": np.arange(1, 7),
    "explained variance": explained_cancer[:6],
    "cumulative variance": np.cumsum(explained_cancer)[:6],
}).round({"explained variance": 4, "cumulative variance": 4}))

# 1.6 Eigenfaces and data-adapted coordinates

If vectorized images form the columns of a matrix $X$, the left singular vectors of the centered matrix are image-shaped basis functions. For face data they are called **eigenfaces**.

The workflow is:

1. Align and vectorize comparable images.
2. Subtract the mean image.
3. Compute the SVD of the centered image matrix.
4. Reshape leading left singular vectors to visualize dominant modes.
5. Represent an image by its coefficients $a=U_q^T(x-\mu)$.
6. Reconstruct with $\hat x=\mu+U_qa$.

The code below uses bundled digit images as a fully self-contained eigenimage example. The mathematics is identical to eigenfaces: a learned low-dimensional basis reconstructs images and supplies coordinates for comparison or classification.

In [ ]:
digit_images = digits_bundle.images
digit_labels = digits_bundle.target
digit_vectors = digit_images.reshape(len(digit_images), -1).T  # pixels × samples

# Hold out one image; learn the basis from all remaining images.
held_out_index = 123
training_mask = np.arange(digit_vectors.shape[1]) != held_out_index
training_vectors = digit_vectors[:, training_mask]
mean_digit = training_vectors.mean(axis=1, keepdims=True)
centered_digits = training_vectors - mean_digit
U_digits, s_digits, Vt_digits = np.linalg.svd(centered_digits, full_matrices=False)

fig, axes = plt.subplots(2, 6, figsize=(12, 4))
axes[0, 0].imshow(mean_digit.reshape(8, 8), cmap="gray")
axes[0, 0].set_title("Mean")
for k in range(1, 6):
    axes[0, k].imshow(U_digits[:, k - 1].reshape(8, 8), cmap="coolwarm")
    axes[0, k].set_title(f"Mode {k}")
sample_indices = [0, 1, 2, 3, 4, 5]
for ax, idx in zip(axes[1], sample_indices):
    ax.imshow(digit_images[idx], cmap="gray")
    ax.set_title(f"Digit {digit_labels[idx]}")
for ax in axes.ravel():
    ax.axis("off")
plt.suptitle("A data-adapted image basis", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
held_out_digit = digit_vectors[:, held_out_index:held_out_index + 1]
centered_held_out = held_out_digit - mean_digit
reconstruction_ranks = [1, 2, 4, 8, 16, 32]
reconstruction_errors = []

fig, axes = plt.subplots(1, len(reconstruction_ranks) + 1, figsize=(14, 2.5))
axes[0].imshow(held_out_digit.reshape(8, 8), cmap="gray", vmin=0, vmax=16)
axes[0].set_title(f"Original\nlabel {digit_labels[held_out_index]}")

for ax, rank in zip(axes[1:], reconstruction_ranks):
    basis = U_digits[:, :rank]
    reconstructed = mean_digit + basis @ (basis.T @ centered_held_out)
    error = np.linalg.norm(held_out_digit - reconstructed) / np.linalg.norm(held_out_digit)
    reconstruction_errors.append(error)
    ax.imshow(reconstructed.reshape(8, 8), cmap="gray", vmin=0, vmax=16)
    ax.set_title(f"rank {rank}\nerr {error:.3f}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Low-dimensional image coordinates for two classes.
all_centered_digits = digit_vectors - mean_digit
all_scores_digits = U_digits.T @ all_centered_digits
class_pair = (2, 7)
pair_mask = np.isin(digit_labels, class_pair)

plt.figure(figsize=(7, 5))
for label, color in zip(class_pair, ["tab:orange", "tab:blue"]):
    mask = digit_labels == label
    plt.scatter(all_scores_digits[4, mask], all_scores_digits[5, mask],
                s=22, alpha=0.65, label=f"digit {label}", color=color)
plt.xlabel("Mode 5 coefficient")
plt.ylabel("Mode 6 coefficient")
plt.title("Images become points in a learned coordinate system")
plt.legend()
plt.show()

# 1.7 SVD for denoising and robustness

Suppose a clean signal matrix is approximately low rank:

$$
X=X_{\text{clean}}+N.
$$

Coherent structure tends to concentrate in a few large singular values, while unstructured noise spreads across many directions. A hard-threshold reconstruction keeps singular values above a threshold $\tau$:

$$
\hat X=\sum_{\sigma_k>\tau}\sigma_ku_kv_k^T.
$$

For a square matrix with independent Gaussian noise of known standard deviation $\eta$, the chapter illustrates the asymptotic threshold

$$
\tau\approx\frac{4}{\sqrt{3}}\sqrt{n}\,\eta.
$$

Threshold formulas depend on matrix shape and noise assumptions; they are not universal. A fixed cumulative-energy rule can also be misleading because noise itself contributes energy.

Low rank is coordinate dependent. A sharply rotated image may require more singular vectors than an axis-aligned version because the matrix rows and columns no longer align with the structure.

In [ ]:
# Rank-2 signal plus Gaussian noise.
n_denoise = 180
t_denoise = np.linspace(0, 2 * np.pi, n_denoise)
clean_matrix = (
    2.5 * np.outer(np.sin(t_denoise), np.cos(2 * t_denoise))
    + 1.3 * np.outer(np.cos(3 * t_denoise), np.sin(t_denoise))
)
noise_sigma = 0.35
noisy_matrix = clean_matrix + noise_sigma * rng.standard_normal(clean_matrix.shape)

U_noisy, s_noisy, Vt_noisy = np.linalg.svd(noisy_matrix, full_matrices=False)
optimal_threshold = (4 / np.sqrt(3)) * np.sqrt(n_denoise) * noise_sigma
rank_threshold = int(np.sum(s_noisy > optimal_threshold))
energy_noisy = np.cumsum(s_noisy**2) / np.sum(s_noisy**2)
rank_90 = int(np.searchsorted(energy_noisy, 0.90) + 1)

def reconstruct_from_svd(U, s, Vt, rank):
    return (U[:, :rank] * s[:rank]) @ Vt[:rank]

denoised_threshold = reconstruct_from_svd(U_noisy, s_noisy, Vt_noisy, rank_threshold)
denoised_90 = reconstruct_from_svd(U_noisy, s_noisy, Vt_noisy, rank_90)

fig, axes = plt.subplots(1, 4, figsize=(15, 3.8))
matrices = [clean_matrix, noisy_matrix, denoised_threshold, denoised_90]
titles = ["Clean rank-2 signal", "Noisy observation",
          f"Hard threshold (rank {rank_threshold})", f"90% energy (rank {rank_90})"]
limit = np.max(np.abs(noisy_matrix))
for ax, matrix, title in zip(axes, matrices, titles):
    ax.imshow(matrix, cmap="coolwarm", vmin=-limit, vmax=limit)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

for name, estimate in [("Noisy", noisy_matrix), ("Threshold", denoised_threshold), ("90% energy", denoised_90)]:
    error = np.linalg.norm(clean_matrix - estimate, "fro") / np.linalg.norm(clean_matrix, "fro")
    print(f"{name:>10} relative error: {error:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].semilogy(s_noisy, "o", ms=3)
axes[0].axhline(optimal_threshold, color="tab:red", ls="--", label=f"threshold = {optimal_threshold:.2f}")
axes[0].set(title="Noisy singular-value spectrum", xlabel="Index", ylabel="$\\sigma_k$")
axes[0].legend()

axes[1].plot(energy_noisy, lw=2)
axes[1].axhline(0.90, color="tab:red", ls="--")
axes[1].axvline(rank_90 - 1, color="tab:gray", ls=":", label=f"rank {rank_90}")
axes[1].set(title="Noise raises cumulative energy", xlabel="Retained modes", ylabel="Fraction", ylim=(0, 1.02))
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# A rotated square generally has a slower singular-value decay.
square_image = np.zeros((140, 140))
square_image[38:102, 46:94] = 1.0
rotation_angles = [0, 10, 20, 30, 40]

fig, axes = plt.subplots(2, len(rotation_angles), figsize=(14, 5.5))
for column, angle_degrees in enumerate(rotation_angles):
    rotated_square = rotate(square_image, angle_degrees, reshape=False, order=1)
    singular_values_rotated = np.linalg.svd(rotated_square, compute_uv=False)
    cumulative_rotated = np.cumsum(singular_values_rotated**2) / np.sum(singular_values_rotated**2)
    rank_99 = np.searchsorted(cumulative_rotated, 0.99) + 1

    axes[0, column].imshow(rotated_square, cmap="gray", vmin=0, vmax=1)
    axes[0, column].set_title(f"{angle_degrees}°")
    axes[0, column].axis("off")
    axes[1, column].semilogy(singular_values_rotated[:35], "o-", ms=2)
    axes[1, column].set_title(f"99% rank = {rank_99}")
    axes[1, column].set_xlabel("Index")
axes[1, 0].set_ylabel("Singular value")
plt.suptitle("Matrix rank depends on alignment with rows and columns", y=1.02)
plt.tight_layout()
plt.show()

# 1.8 Randomized SVD

A full SVD may be too expensive when $X$ is very large and only the leading $r$ modes are needed. Randomized SVD first finds a low-dimensional subspace that approximately contains the columns of $X$.

A basic algorithm is:

1. Draw a random test matrix $\Omega\in\mathbb{R}^{n\times(r+p)}$, where $p$ is oversampling.
2. Form $Y=X\Omega$.
3. Optionally apply power iterations, $Y\leftarrow X(X^TY)$, to separate the dominant spectrum.
4. Orthonormalize: $Y=QR$.
5. Compute the small matrix $B=Q^TX$.
6. Compute $B=\tilde U\Sigma V^T$ and set $U=Q\tilde U$.

Oversampling makes it less likely that the random sketch misses an important direction. Power iterations improve accuracy when singular values decay slowly, but require more passes through the data.

In [ ]:
def randomized_svd(X, rank, oversampling=8, power_iterations=1, seed=0):
    """Approximate the leading singular triplets using a randomized range finder."""
    local_rng = np.random.default_rng(seed)
    sketch_size = min(rank + oversampling, min(X.shape))
    omega = local_rng.standard_normal((X.shape[1], sketch_size))
    Y = X @ omega

    # QR between power steps limits numerical loss of orthogonality.
    for _ in range(power_iterations):
        Y, _ = np.linalg.qr(Y, mode="reduced")
        Y = X @ (X.T @ Y)

    Q, _ = np.linalg.qr(Y, mode="reduced")
    B = Q.T @ X
    U_small, s, Vt = np.linalg.svd(B, full_matrices=False)
    U = Q @ U_small
    return U[:, :rank], s[:rank], Vt[:rank]

In [ ]:
randomized_rank = 20
U_random, s_random, Vt_random = randomized_svd(
    digit_mosaic, randomized_rank, oversampling=10, power_iterations=1, seed=SEED
)
randomized_approximation = (U_random * s_random) @ Vt_random
deterministic_approximation = (U_image[:, :randomized_rank] * s_image[:randomized_rank]) @ Vt_image[:randomized_rank]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, matrix, title in zip(
    axes,
    [digit_mosaic, deterministic_approximation, randomized_approximation],
    ["Original", "Deterministic rank-20 SVD", "Randomized rank-20 SVD"],
):
    ax.imshow(matrix, cmap="gray", vmin=0, vmax=16)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

deterministic_error = np.linalg.norm(digit_mosaic - deterministic_approximation, "fro")
randomized_error = np.linalg.norm(digit_mosaic - randomized_approximation, "fro")
print(f"Deterministic error: {deterministic_error:.4f}")
print(f"Randomized error:    {randomized_error:.4f}")
print(f"Error ratio:         {randomized_error / deterministic_error:.4f}")

In [ ]:
# Power iterations help when the spectrum decays slowly.
spectrum = np.geomspace(1.0, 0.05, 120)
Q1, _ = np.linalg.qr(rng.standard_normal((180, 120)))
Q2, _ = np.linalg.qr(rng.standard_normal((150, 120)))
slow_decay_matrix = (Q1 * spectrum) @ Q2.T
target_rank = 12
optimal_error = spectrum[target_rank]  # best rank-r spectral-norm error

power_rows = []
for q in range(4):
    U_q, s_q, Vt_q = randomized_svd(
        slow_decay_matrix, target_rank, oversampling=5, power_iterations=q, seed=SEED
    )
    approximation_q = (U_q * s_q) @ Vt_q
    error_q = np.linalg.norm(slow_decay_matrix - approximation_q, 2)
    power_rows.append({
        "power iterations": q,
        "spectral error": error_q,
        "error / optimum": error_q / optimal_error,
    })

power_results = pd.DataFrame(power_rows)
display(power_results.round({"spectral error": 5, "error / optimum": 3}))

plt.figure(figsize=(7, 4))
plt.plot(power_results["power iterations"], power_results["error / optimum"], "o-", lw=2)
plt.axhline(1, color="black", ls="--", label="Best possible rank-12 error")
plt.xticks(power_results["power iterations"])
plt.xlabel("Power iterations")
plt.ylabel("Error relative to optimum")
plt.title("Power iterations sharpen the randomized subspace")
plt.legend()
plt.show()

# 1.9 Tensors and higher-order data

A tensor is a multidimensional array. A video, for example, can be stored as height $\times$ width $\times$ time. Flattening a tensor into a matrix is called **unfolding** or **matricization**.

Two common tensor models are:

- **CP/PARAFAC:** a sum of rank-one outer products,

  $$
  \mathcal X\approx\sum_{k=1}^{r}a_k\circ b_k\circ c_k.
  $$

- **Tucker/HOSVD:** a small core tensor multiplied by low-dimensional factor matrices along each mode,

  $$
  \mathcal X\approx\mathcal G\times_1U_1\times_2U_2\times_3U_3.
  $$

Unlike matrix rank, tensor rank has several definitions, and a globally best low-rank tensor approximation is generally harder to compute. HOSVD provides a practical multilinear analogue of the SVD by taking leading left singular vectors from each unfolding.

In [ ]:
# Synthetic space-time field built from two separable components.
y_tensor = np.linspace(-2, 2, 48)
x_tensor = np.linspace(-3, 3, 64)
t_tensor = np.linspace(0, 4 * np.pi, 80)

y_mode_1 = np.exp(-1.2 * y_tensor**2)
x_mode_1 = np.sin(1.5 * x_tensor)
time_mode_1 = np.cos(t_tensor)

y_mode_2 = y_tensor * np.exp(-0.8 * y_tensor**2)
x_mode_2 = np.cos(0.8 * x_tensor)
time_mode_2 = 0.7 * np.sin(2 * t_tensor)

tensor_field = (
    np.einsum("i,j,k->ijk", y_mode_1, x_mode_1, time_mode_1)
    + np.einsum("i,j,k->ijk", y_mode_2, x_mode_2, time_mode_2)
)

snapshot_indices = [0, 10, 20, 30, 40]
fig, axes = plt.subplots(1, len(snapshot_indices), figsize=(15, 3))
field_limit = np.max(np.abs(tensor_field))
for ax, index in zip(axes, snapshot_indices):
    ax.imshow(tensor_field[:, :, index], cmap="coolwarm", origin="lower",
              extent=[x_tensor.min(), x_tensor.max(), y_tensor.min(), y_tensor.max()],
              vmin=-field_limit, vmax=field_limit, aspect="auto")
    ax.set_title(f"time index {index}")
    ax.set_xlabel("x")
axes[0].set_ylabel("y")
plt.suptitle("Snapshots of a low-rank space-time tensor", y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
def hosvd_rank(tensor, ranks):
    """Truncated HOSVD for a three-way tensor."""
    rank_y, rank_x, rank_t = ranks
    unfold_y = tensor.reshape(tensor.shape[0], -1)
    unfold_x = tensor.transpose(1, 0, 2).reshape(tensor.shape[1], -1)
    unfold_t = tensor.transpose(2, 0, 1).reshape(tensor.shape[2], -1)

    U_y = np.linalg.svd(unfold_y, full_matrices=False)[0][:, :rank_y]
    U_x = np.linalg.svd(unfold_x, full_matrices=False)[0][:, :rank_x]
    U_t = np.linalg.svd(unfold_t, full_matrices=False)[0][:, :rank_t]

    core = np.einsum("ia,jb,kc,ijk->abc", U_y, U_x, U_t, tensor)
    reconstruction = np.einsum("ia,jb,kc,abc->ijk", U_y, U_x, U_t, core)
    return reconstruction, core, (U_y, U_x, U_t)

tensor_reconstruction, tensor_core, tensor_factors = hosvd_rank(tensor_field, (2, 2, 2))
tensor_error = np.linalg.norm(tensor_field - tensor_reconstruction) / np.linalg.norm(tensor_field)
U_y_tensor, U_x_tensor, U_t_tensor = tensor_factors

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(y_tensor, U_y_tensor)
axes[0].set(title="Leading y-mode factors", xlabel="y")
axes[1].plot(x_tensor, U_x_tensor)
axes[1].set(title="Leading x-mode factors", xlabel="x")
axes[2].plot(t_tensor, U_t_tensor)
axes[2].set(title="Leading time-mode factors", xlabel="t")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
index = 17
comparison = [tensor_field[:, :, index], tensor_reconstruction[:, :, index],
              tensor_field[:, :, index] - tensor_reconstruction[:, :, index]]
titles = ["Original snapshot", "Rank-(2,2,2) HOSVD", "Residual"]
for ax, matrix, title in zip(axes, comparison, titles):
    ax.imshow(matrix, cmap="coolwarm", origin="lower", vmin=-field_limit, vmax=field_limit)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

print("Core tensor shape:", tensor_core.shape)
print(f"Relative reconstruction error: {tensor_error:.2e}")

# Chapter 1 roadmap and review checklist

| Topic | Core object | What to remember |
|---|---|---|
| SVD geometry | $X=U\Sigma V^T$ | Rotate, scale, rotate |
| Low-rank approximation | $X_q=U_q\Sigma_qV_q^T$ | Truncated SVD is optimal for the 2- and Frobenius norms |
| Correlation matrices | $XX^T$ and $X^TX$ | Eigenvalues are $\sigma_k^2$ |
| Pseudoinverse | $X^+=V\Sigma^+U^T$ | Solves minimum-norm least squares |
| PCA | centered-data SVD | Loadings are $V$; scores are $U\Sigma$ |
| Eigenimages | SVD of an image ensemble | Learn a basis from the data itself |
| Denoising | singular-value thresholding | Retain coherent dominant modes |
| Randomized SVD | random range sketch | Approximate leading modes with fewer computations |
| Tensors | CP or Tucker/HOSVD | Preserve multiway structure instead of flattening everything once |

## Questions to answer after running the notebook

1. Why do squared singular values, rather than singular values, determine explained variance?
2. At what rank does image compression stop producing a visible improvement?
3. Why can a small singular value make regression coefficients unstable?
4. What information is lost when PCA retains only the first few scores?
5. Why does rotating a simple image change its matrix singular-value spectrum?
6. How do oversampling and power iterations affect randomized SVD?
7. What does each factor matrix in the HOSVD describe?

End-of-chapter problem solutions are intentionally not included; this notebook covers the chapter narrative and its computational examples.